In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
label_registry = pd.read_csv(
    REPORTS_DIR
    / "dataset_label_registry.csv"
)

source_feasibility = pd.read_csv(
    REPORTS_DIR
    / "source_feature_feasibility.csv"
)

catching_final = pd.read_csv(
    REPORTS_DIR
    / "catching_repository_final.csv"
)

catching_targets = pd.read_csv(
    INTERIM_DIR
    / "catching_targets.csv"
)

print(
    "Label registry:",
    label_registry.shape
)

print(
    "Source feasibility:",
    source_feasibility.shape
)

print(
    "Catching repositories:",
    catching_final.shape
)

print(
    "Catching targets:",
    catching_targets.shape
)

Label registry: (10, 10)
Source feasibility: (10, 9)
Catching repositories: (30, 3)
Catching targets: (30, 8)


In [3]:
catching_repository_counts = (
    catching_final[
        "repository_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "repository_status"
    )
    .reset_index(
        name="count"
    )
)

catching_repository_counts

,repository_status,count
0,UNRESOLVED,21
1,PROVISIONAL_MODERATE,5
2,AMBIGUOUS_MULTIPLE_MODERATE,2
3,CONFIRMED_STRONG,2


In [4]:
n_strong = (
    catching_final[
        "repository_status"
    ]
    ==
    "CONFIRMED_STRONG"
).sum()

n_moderate = (
    catching_final[
        "repository_status"
    ]
    ==
    "PROVISIONAL_MODERATE"
).sum()

n_ambiguous = (
    catching_final[
        "repository_status"
    ]
    .astype(str)
    .str.startswith(
        "AMBIGUOUS"
    )
).sum()

n_unresolved = (
    catching_final[
        "repository_status"
    ]
    ==
    "UNRESOLVED"
).sum()

print(
    "Strong:",
    n_strong
)

print(
    "Moderate:",
    n_moderate
)

print(
    "Ambiguous:",
    n_ambiguous
)

print(
    "Unresolved:",
    n_unresolved
)

Strong: 2
Moderate: 5
Ambiguous: 2
Unresolved: 21


In [5]:
repository_usage_policy = pd.DataFrame([
    {
        "repository_status":
            "CONFIRMED_STRONG",

        "allowed_for_primary_X":
            True,

        "allowed_for_secondary_X":
            True,

        "interpretation":
            "Repositorio identificado mediante evidencia textual fuerte."
    },

    {
        "repository_status":
            "PROVISIONAL_MODERATE",

        "allowed_for_primary_X":
            False,

        "allowed_for_secondary_X":
            True,

        "interpretation":
            "Existe coincidencia, pero no evidencia suficiente para considerarlo fuente confirmada."
    },

    {
        "repository_status":
            "AMBIGUOUS_MULTIPLE_MODERATE",

        "allowed_for_primary_X":
            False,

        "allowed_for_secondary_X":
            False,

        "interpretation":
            "Más de un repositorio es compatible con la evidencia."
    },

    {
        "repository_status":
            "UNRESOLVED",

        "allowed_for_primary_X":
            False,

        "allowed_for_secondary_X":
            False,

        "interpretation":
            "No se pudo identificar con suficiente evidencia el código evaluado."
    }
])

repository_usage_policy

,repository_status,allowed_for_primary_X,allowed_for_secondary_X,interpretation
0,CONFIRMED_STRONG,True,True,Repositorio identificado mediante evidencia textual fuerte.
1,PROVISIONAL_MODERATE,False,True,"Existe coincidencia, pero no evidencia suficiente para considerarlo fuente confirmada."
2,AMBIGUOUS_MULTIPLE_MODERATE,False,False,Más de un repositorio es compatible con la evidencia.
3,UNRESOLVED,False,False,No se pudo identificar con suficiente evidencia el código evaluado.


In [6]:
target_source_matrix = pd.DataFrame([
    {
        "dataset": "Catching UX Flaws",
        "H1": "DIRECT_LLM",
        "H4": "DIRECT_LLM",
        "H5": "DIRECT_LLM",
        "H6": "DIRECT_LLM",
        "X_type": "WEB_SOURCE",
        "usable_instances_now": n_strong,
        "role": "DIRECT_VALIDATION_SMALL",
        "primary_training": False,
        "reason":
            "Y Nielsen directa, pero solo dos repositorios tienen X fuertemente confirmada."
    },

    {
        "dataset": "Dashboards",
        "H1": None,
        "H4": "EXPERIMENTAL_PROXY",
        "H5": None,
        "H6": "EXPERIMENTAL_PROXY",
        "X_type": "BEHAVIOR_INTERACTION",
        "usable_instances_now": 1008,
        "role": "PROXY_TRAINING",
        "primary_training": False,
        "reason":
            "X conductual abundante; las condiciones experimentales no son Nielsen directo."
    },

    {
        "dataset": "UICrit + RICO",
        "H1": None,
        "H4": "RATING_PROXY",
        "H5": None,
        "H6": "RATING_PROXY",
        "X_type": "UI_HIERARCHY",
        "usable_instances_now": 1000,
        "role": "PROXY_TRAINING",
        "primary_training": False,
        "reason":
            "X estructural con join completo; ratings humanos relacionados, no Nielsen directo."
    },

    {
        "dataset": "29042675",
        "H1": "DIRECT_CANDIDATE",
        "H4": "DIRECT_CANDIDATE",
        "H5": "DIRECT_CANDIDATE",
        "H6": "DIRECT_CANDIDATE",
        "X_type": "PROTOTYPE",
        "usable_instances_now": 0,
        "role": "DIRECT_PENDING_X",
        "primary_training": False,
        "reason":
            "Y heurística disponible, pero X equivalente todavía no recuperada."
    },

    {
        "dataset": "AMUSED",
        "H1": "PROXY_CANDIDATE",
        "H4": None,
        "H5": "PROXY_CANDIDATE",
        "H6": "PROXY_CANDIDATE",
        "X_type": "BEHAVIOR_INTERACTION",
        "usable_instances_now": 0,
        "role": "PENDING_ACCESS",
        "primary_training": False,
        "reason":
            "Fuente potencialmente valiosa pero acceso todavía pendiente."
    },

    {
        "dataset": "P3M",
        "H1": "LLM_DIRECT_POSITIVE",
        "H4": "LLM_DIRECT_POSITIVE",
        "H5": "LLM_DIRECT_POSITIVE",
        "H6": "LLM_DIRECT_POSITIVE",
        "X_type": "SCREENSHOT",
        "usable_instances_now": 20,
        "role": "SECONDARY_VISUAL",
        "primary_training": False,
        "reason":
            "No existe equivalencia clara entre screenshot features y las X DOM/telemetría de la extensión."
    },

    {
        "dataset": "PHR",
        "H1": "HUMAN_PROXY",
        "H4": "HUMAN_PROXY",
        "H5": "HUMAN_PROXY",
        "H6": "HUMAN_PROXY",
        "X_type": "TASK_AGGREGATES",
        "usable_instances_now": 0,
        "role": "REFERENCE_PROXY",
        "primary_training": False,
        "reason":
            "Las ocho heurísticas son específicas de PHR y no deben mapearse numéricamente a Nielsen."
    },

    {
        "dataset": "SERENE",
        "H1": None,
        "H4": None,
        "H5": None,
        "H6": None,
        "X_type": "METRICS",
        "usable_instances_now": 0,
        "role": "LABEL_REFERENCE",
        "primary_training": False,
        "reason":
            "Problemas humanos, pero sin asignación explícita a nuestras cuatro heurísticas."
    }
])

display(
    target_source_matrix
)

,dataset,H1,H4,H5,H6,X_type,usable_instances_now,role,primary_training,reason
0,Catching UX Flaws,DIRECT_LLM,DIRECT_LLM,DIRECT_LLM,DIRECT_LLM,WEB_SOURCE,2,DIRECT_VALIDATION_SMALL,False,"Y Nielsen directa, pero solo dos repositorios tienen X fuertemente confirmada."
1,Dashboards,NaN,EXPERIMENTAL_PROXY,NaN,EXPERIMENTAL_PROXY,BEHAVIOR_INTERACTION,1008,PROXY_TRAINING,False,X conductual abundante; las condiciones experimentales no son Nielsen directo.
2,UICrit + RICO,NaN,RATING_PROXY,NaN,RATING_PROXY,UI_HIERARCHY,1000,PROXY_TRAINING,False,"X estructural con join completo; ratings humanos relacionados, no Nielsen directo."
3,29042675,DIRECT_CANDIDATE,DIRECT_CANDIDATE,DIRECT_CANDIDATE,DIRECT_CANDIDATE,PROTOTYPE,0,DIRECT_PENDING_X,False,"Y heurística disponible, pero X equivalente todavía no recuperada."
4,AMUSED,PROXY_CANDIDATE,NaN,PROXY_CANDIDATE,PROXY_CANDIDATE,BEHAVIOR_INTERACTION,0,PENDING_ACCESS,False,Fuente potencialmente valiosa pero acceso todavía pendiente.
5,P3M,LLM_DIRECT_POSITIVE,LLM_DIRECT_POSITIVE,LLM_DIRECT_POSITIVE,LLM_DIRECT_POSITIVE,SCREENSHOT,20,SECONDARY_VISUAL,False,No existe equivalencia clara entre screenshot features y las X DOM/telemetría de la extensión.
6,PHR,HUMAN_PROXY,HUMAN_PROXY,HUMAN_PROXY,HUMAN_PROXY,TASK_AGGREGATES,0,REFERENCE_PROXY,False,Las ocho heurísticas son específicas de PHR y no deben mapearse numéricamente a Nielsen.
7,SERENE,NaN,NaN,NaN,NaN,METRICS,0,LABEL_REFERENCE,False,"Problemas humanos, pero sin asignación explícita a nuestras cuatro heurísticas."


In [7]:
target_viability = pd.DataFrame([
    {
        "target": "H1_violation",
        "direct_source_ready": False,
        "direct_source": "Catching UX Flaws",
        "direct_instances_confirmed": n_strong,
        "proxy_sources": "AMUSED",
        "current_viability": "INSUFFICIENT_DIRECT_DATA"
    },

    {
        "target": "H4_violation",
        "direct_source_ready": False,
        "direct_source": "Catching UX Flaws",
        "direct_instances_confirmed": n_strong,
        "proxy_sources": "Dashboards | UICrit+RICO",
        "current_viability": "PROXY_DATA_AVAILABLE"
    },

    {
        "target": "H5_violation",
        "direct_source_ready": False,
        "direct_source": "Catching UX Flaws",
        "direct_instances_confirmed": n_strong,
        "proxy_sources": "AMUSED",
        "current_viability": "INSUFFICIENT_DIRECT_DATA"
    },

    {
        "target": "H6_violation",
        "direct_source_ready": False,
        "direct_source": "Catching UX Flaws",
        "direct_instances_confirmed": n_strong,
        "proxy_sources": "Dashboards | UICrit+RICO | AMUSED",
        "current_viability": "PROXY_DATA_AVAILABLE"
    }
])

target_viability

,target,direct_source_ready,direct_source,direct_instances_confirmed,proxy_sources,current_viability
0,H1_violation,False,Catching UX Flaws,2,AMUSED,INSUFFICIENT_DIRECT_DATA
1,H4_violation,False,Catching UX Flaws,2,Dashboards | UICrit+RICO,PROXY_DATA_AVAILABLE
2,H5_violation,False,Catching UX Flaws,2,AMUSED,INSUFFICIENT_DIRECT_DATA
3,H6_violation,False,Catching UX Flaws,2,Dashboards | UICrit+RICO | AMUSED,PROXY_DATA_AVAILABLE


In [8]:
experiment_registry = pd.DataFrame([
    {
        "experiment": "E1_DIRECT",
        "allowed_labels":
            "HUMAN_DIRECT | LLM_DIRECT",
        "purpose":
            "Evaluación principal cuando exista suficiente X+Y directa."
    },

    {
        "experiment": "E2_PROXY",
        "allowed_labels":
            "EXPERIMENTAL_PROXY | RATING_PROXY",
        "purpose":
            "Evaluar si constructos relacionados permiten aprender señales transferibles."
    },

    {
        "experiment": "E3_DIRECT_PLUS_PROXY",
        "allowed_labels":
            "DIRECT + PROXY",
        "purpose":
            "Análisis de sensibilidad; nunca reportarlo como equivalente a GOLD."
    }
])

experiment_registry

,experiment,allowed_labels,purpose
0,E1_DIRECT,HUMAN_DIRECT | LLM_DIRECT,Evaluación principal cuando exista suficiente X+Y directa.
1,E2_PROXY,EXPERIMENTAL_PROXY | RATING_PROXY,Evaluar si constructos relacionados permiten aprender señales transferibles.
2,E3_DIRECT_PLUS_PROXY,DIRECT + PROXY,Análisis de sensibilidad; nunca reportarlo como equivalente a GOLD.


In [9]:
catching_decision = pd.DataFrame([
    {
        "use_case":
            "PRIMARY_TRAINING",

        "decision":
            "NO",

        "reason":
            "Solo dos interfaces poseen repositorio confirmado con evidencia fuerte."
    },

    {
        "use_case":
            "SECONDARY_TRAINING",

        "decision":
            "CONDITIONAL",

        "reason":
            "Puede ampliarse con cinco repositorios PROVISIONAL_MODERATE, manteniendo provenance separada."
    },

    {
        "use_case":
            "PIPELINE_VALIDATION",

        "decision":
            "YES",

        "reason":
            "Los dos CONFIRMED_STRONG permiten comprobar extracción X + unión con Nielsen Y."
    },

    {
        "use_case":
            "EXTERNAL_CASE_STUDY",

        "decision":
            "YES",

        "reason":
            "Puede documentarse como evidencia de factibilidad, no como conjunto estadísticamente suficiente."
    }
])

catching_decision

,use_case,decision,reason
0,PRIMARY_TRAINING,NO,Solo dos interfaces poseen repositorio confirmado con evidencia fuerte.
1,SECONDARY_TRAINING,CONDITIONAL,"Puede ampliarse con cinco repositorios PROVISIONAL_MODERATE, manteniendo provenance separada."
2,PIPELINE_VALIDATION,YES,Los dos CONFIRMED_STRONG permiten comprobar extracción X + unión con Nielsen Y.
3,EXTERNAL_CASE_STUDY,YES,"Puede documentarse como evidencia de factibilidad, no como conjunto estadísticamente suficiente."


In [10]:
adapter_priority = pd.DataFrame([
    {
        "priority": 1,
        "dataset": "Dashboards",
        "adapter": "behavior_adapter",
        "why":
            "Gran volumen de telemetría ya parseada y compatible con métricas conductuales de la extensión."
    },

    {
        "priority": 2,
        "dataset": "UICrit + RICO",
        "adapter": "structural_adapter",
        "why":
            "Join de RICO confirmado y fuente estructural reproducible."
    },

    {
        "priority": 3,
        "dataset": "Catching UX Flaws",
        "adapter": "static_validation_adapter",
        "why":
            "Solo para los dos repositorios CONFIRMED_STRONG como validación del pipeline."
    },

    {
        "priority": 4,
        "dataset": "29042675",
        "adapter": "prototype_recovery",
        "why":
            "Puede aportar Nielsen directo si se logra recuperar una X compatible."
    },

    {
        "priority": 5,
        "dataset": "AMUSED",
        "adapter": "behavior_smell_adapter",
        "why":
            "Esperar acceso."
    }
])

adapter_priority

,priority,dataset,adapter,why
0,1,Dashboards,behavior_adapter,Gran volumen de telemetría ya parseada y compatible con métricas conductuales de la extensión.
1,2,UICrit + RICO,structural_adapter,Join de RICO confirmado y fuente estructural reproducible.
2,3,Catching UX Flaws,static_validation_adapter,Solo para los dos repositorios CONFIRMED_STRONG como validación del pipeline.
3,4,29042675,prototype_recovery,Puede aportar Nielsen directo si se logra recuperar una X compatible.
4,5,AMUSED,behavior_smell_adapter,Esperar acceso.


In [11]:
repository_usage_policy.to_csv(
    REPORTS_DIR
    / "repository_usage_policy.csv",
    index=False
)

target_source_matrix.to_csv(
    REPORTS_DIR
    / "target_source_matrix.csv",
    index=False
)

target_viability.to_csv(
    REPORTS_DIR
    / "target_viability.csv",
    index=False
)

experiment_registry.to_csv(
    REPORTS_DIR
    / "experiment_registry.csv",
    index=False
)

catching_decision.to_csv(
    REPORTS_DIR
    / "catching_final_usage_decision.csv",
    index=False
)

adapter_priority.to_csv(
    REPORTS_DIR
    / "dataset_adapter_priority.csv",
    index=False
)

In [12]:
expected = [
    "repository_usage_policy.csv",
    "target_source_matrix.csv",
    "target_viability.csv",
    "experiment_registry.csv",
    "catching_final_usage_decision.csv",
    "dataset_adapter_priority.csv"
]

for filename in expected:

    path = (
        REPORTS_DIR
        / filename
    )

    print(
        filename,
        "✅" if path.exists()
        else "❌"
    )

repository_usage_policy.csv ✅
target_source_matrix.csv ✅
target_viability.csv ✅
experiment_registry.csv ✅
catching_final_usage_decision.csv ✅
dataset_adapter_priority.csv ✅
